# DQN PER

## Pruebas

In [ ]:
import sys
import importlib

import numpy as np

sys.path.append("../src")

import replay_buffer
importlib.reload(replay_buffer)

from replay_buffer import SumTree

arbol_prueba = SumTree(capacidad=4)

prioridades = [1.0, 2.0, 3.0, 4.0]

for indice, prioridad in enumerate(prioridades):
    arbol_prueba.actualizar(
        indice_dato=indice,
        prioridad=prioridad,
    )

print(
    f"Prioridad total esperada: "
    f"{sum(prioridades):.2f}"
)
print(
    f"Prioridad total obtenida: "
    f"{arbol_prueba.prioridad_total:.2f}"
)

print("\nMUESTREO POR VALOR ACUMULADO")

valores_prueba = [0.5, 1.5, 4.5, 8.0]

for valor in valores_prueba:
    indice, prioridad = arbol_prueba.obtener(valor)

    print(
        f"Valor {valor:>4.1f} -> "
        f"índice {indice}, "
        f"prioridad {prioridad:.1f}"
    )

arbol_prueba.actualizar(
    indice_dato=0,
    prioridad=5.0,
)

print(
    "\nPrioridad total después de cambiar "
    f"la prioridad 0 de 1 a 5: "
    f"{arbol_prueba.prioridad_total:.2f}"
)

assert np.isclose(
    arbol_prueba.prioridad_total,
    14.0,
)

print("\nEl SumTree funciona correctamente.")

Prioridad total esperada: 10.00
Prioridad total obtenida: 10.00

MUESTREO POR VALOR ACUMULADO
Valor  0.5 -> índice 0, prioridad 1.0
Valor  1.5 -> índice 1, prioridad 2.0
Valor  4.5 -> índice 2, prioridad 3.0
Valor  8.0 -> índice 3, prioridad 4.0

Prioridad total después de cambiar la prioridad 0 de 1 a 5: 14.00

El SumTree funciona correctamente.


In [ ]:
import torch

importlib.reload(replay_buffer)

from replay_buffer import (
    SumTree,
    PrioritizedReplayBuffer,
)

buffer_per_prueba = PrioritizedReplayBuffer(
    capacidad=8,
    forma_observacion=(4, 84, 84),
    seed=42,
    alpha=0.6,
    epsilon_per=1e-6,
)

for indice in range(8):
    observacion = np.full(
        (4, 84, 84),
        fill_value=indice,
        dtype=np.uint8,
    )

    siguiente_observacion = np.full(
        (4, 84, 84),
        fill_value=indice + 1,
        dtype=np.uint8,
    )

    buffer_per_prueba.agregar(
        observacion=observacion,
        accion=indice % 6,
        recompensa=float(indice),
        siguiente_observacion=siguiente_observacion,
        finalizado=False,
    )

errores_td = np.array(
    [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 10.0],
    dtype=np.float32,
)

buffer_per_prueba.actualizar_prioridades(
    indices=np.arange(8),
    errores_td=errores_td,
)

conteos = np.zeros(8, dtype=np.int64)

for _ in range(500):
    (
        observaciones,
        acciones,
        recompensas,
        siguientes_observaciones,
        finalizados,
        indices,
        pesos,
    ) = buffer_per_prueba.muestrear(
        batch_size=4,
        device=torch.device("cpu"),
        beta=0.4,
    )

    for indice in indices:
        conteos[indice] += 1

print("FRECUENCIA DE MUESTREO")

for indice, conteo in enumerate(conteos):
    print(
        f"Experiencia {indice}: "
        f"{conteo} apariciones"
    )

print(
    "\nPrioridad total: "
    f"{buffer_per_prueba.arbol_prioridades.prioridad_total:.4f}"
)

print(
    "Forma de los pesos: "
    f"{pesos.shape}"
)

print(
    "Rango de los pesos: "
    f"{pesos.min().item():.4f} - "
    f"{pesos.max().item():.4f}"
)

assert len(buffer_per_prueba) == 8
assert pesos.shape == (4,)
assert torch.isfinite(pesos).all()
assert pesos.max() <= 1.0 + 1e-6

assert conteos[7] > conteos[:7].max()

print(
    "\nLa experiencia con mayor error TD fue "
    "muestreada con mayor frecuencia."
)

FRECUENCIA DE MUESTREO
Experiencia 0: 97 apariciones
Experiencia 1: 89 apariciones
Experiencia 2: 80 apariciones
Experiencia 3: 85 apariciones
Experiencia 4: 90 apariciones
Experiencia 5: 79 apariciones
Experiencia 6: 79 apariciones
Experiencia 7: 1401 apariciones

Prioridad total: 5.7394
Forma de los pesos: torch.Size([4])
Rango de los pesos: 0.3311 - 1.0000

La experiencia con mayor error TD fue muestreada con mayor frecuencia.


In [ ]:
from copy import deepcopy

import models
import train

importlib.reload(models)
importlib.reload(train)

from models import DQN
from train import (
    ConfigDQN,
    actualizar_modelo,
)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

config_prueba_per = ConfigDQN(
    batch_size=32,
    capacidad_buffer=64,
    usar_per=True,
    per_alpha=0.6,
    per_beta_inicial=0.4,
    per_beta_final=1.0,
    per_pasos_beta=1_000,
)

modelo_online_prueba = DQN(
    n_acciones=6
).to(device)

modelo_target_prueba = deepcopy(
    modelo_online_prueba
).to(device)

modelo_target_prueba.eval()

optimizador_prueba = torch.optim.Adam(
    modelo_online_prueba.parameters(),
    lr=config_prueba_per.learning_rate,
)

buffer_actualizacion_per = PrioritizedReplayBuffer(
    capacidad=64,
    forma_observacion=(4, 84, 84),
    seed=42,
    alpha=config_prueba_per.per_alpha,
    epsilon_per=config_prueba_per.per_epsilon,
)

rng_prueba = np.random.default_rng(42)

for _ in range(64):
    observacion = rng_prueba.integers(
        0,
        256,
        size=(4, 84, 84),
        dtype=np.uint8,
    )

    siguiente_observacion = rng_prueba.integers(
        0,
        256,
        size=(4, 84, 84),
        dtype=np.uint8,
    )

    buffer_actualizacion_per.agregar(
        observacion=observacion,
        accion=int(rng_prueba.integers(6)),
        recompensa=float(
            rng_prueba.choice([-1.0, 0.0, 1.0])
        ),
        siguiente_observacion=siguiente_observacion,
        finalizado=bool(
            rng_prueba.random() < 0.1
        ),
    )

inicio_hojas = (
    buffer_actualizacion_per.capacidad - 1
)

prioridades_antes = (
    buffer_actualizacion_per
    .arbol_prioridades
    .arbol[
        inicio_hojas:
        inicio_hojas + len(buffer_actualizacion_per)
    ]
    .copy()
)

metricas_prueba_per = actualizar_modelo(
    modelo_online=modelo_online_prueba,
    modelo_target=modelo_target_prueba,
    replay_buffer=buffer_actualizacion_per,
    optimizador=optimizador_prueba,
    config=config_prueba_per,
    device=device,
    usar_double_dqn=False,
    paso_global=500,
)

prioridades_despues = (
    buffer_actualizacion_per
    .arbol_prioridades
    .arbol[
        inicio_hojas:
        inicio_hojas + len(buffer_actualizacion_per)
    ]
    .copy()
)

prioridades_modificadas = np.count_nonzero(
    ~np.isclose(
        prioridades_antes,
        prioridades_despues,
    )
)

print(f"Dispositivo: {device}")
print(
    f"Loss: "
    f"{metricas_prueba_per['loss']:.6f}"
)
print(
    f"Error TD promedio: "
    f"{metricas_prueba_per['error_td_promedio']:.6f}"
)
print(
    f"Beta utilizado: "
    f"{metricas_prueba_per['beta_per']:.3f}"
)
print(
    f"Peso de importancia promedio: "
    f"{metricas_prueba_per['peso_importancia_promedio']:.3f}"
)
print(
    f"Prioridades modificadas: "
    f"{prioridades_modificadas}"
)

assert np.isfinite(
    metricas_prueba_per["loss"]
)

assert prioridades_modificadas > 0

assert np.isclose(
    metricas_prueba_per["beta_per"],
    0.7,
)

print(
    "\nLa actualización con PER funciona correctamente."
)

Dispositivo: mps
Loss: 0.214474
Error TD promedio: 0.449528
Beta utilizado: 0.700
Peso de importancia promedio: 1.000
Prioridades modificadas: 32

La actualización con PER funciona correctamente.


In [4]:
import gc
import pandas as pd
from pathlib import Path

importlib.reload(train)

from train import (
    ConfigDQN,
    entrenar_dqn,
)

objetos_temporales = [
    "buffer_per_prueba",
    "buffer_actualizacion_per",
    "modelo_online_prueba",
    "modelo_target_prueba",
    "optimizador_prueba",
]

for nombre in objetos_temporales:
    globals().pop(nombre, None)

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

config_smoke_per = ConfigDQN(
    nombre_experimento="smoke_test_dqn_per",
    total_pasos=2_000,

    usar_per=True,
    per_alpha=0.6,
    per_beta_inicial=0.4,
    per_beta_final=1.0,
    per_pasos_beta=2_000,
    per_epsilon=1e-6,

    capacidad_buffer=2_000,
    inicio_entrenamiento=200,
    batch_size=32,
    frecuencia_entrenamiento=4,
    frecuencia_actualizacion_target=500,

    epsilon_inicial=1.0,
    epsilon_final=0.1,
    pasos_decay_epsilon=1_000,

    frecuencia_log=250,
    frecuencia_evaluacion=2_000,
    episodios_evaluacion=1,
)

resultado_smoke_per = entrenar_dqn(
    config=config_smoke_per,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
)

print("\nSMOKE TEST DQN + PER FINALIZADO")
print(
    "Mejor promedio de evaluación: "
    f"{resultado_smoke_per['mejor_promedio_evaluacion']:.2f}"
)

ruta_updates_smoke = Path(
    "../logs/entrenamientos/"
    "smoke_test_dqn_per/actualizaciones.csv"
)

df_updates_smoke = pd.read_csv(
    ruta_updates_smoke
)

display(
    df_updates_smoke[
        [
            "paso_global",
            "loss",
            "error_td_promedio",
            "beta_per",
            "peso_importancia_promedio",
            "tamano_buffer",
        ]
    ]
)

A.L.E: Arcade Learning Environment (version 0.10.1+6a7e0ae)
[Powered by Stella]


Replay buffer: priorizado | alpha=0.6 | beta inicial=0.4
Paso 250/2,000 | episodio=1 | epsilon=0.775 | loss=0.0054 | Q=0.071
Paso 500/2,000 | episodio=2 | epsilon=0.550 | loss=0.0133 | Q=0.103
Paso 750/2,000 | episodio=3 | epsilon=0.325 | loss=0.0021 | Q=0.185
Paso 1,000/2,000 | episodio=5 | epsilon=0.100 | loss=0.0039 | Q=0.177
Paso 1,250/2,000 | episodio=6 | epsilon=0.100 | loss=0.0104 | Q=0.297
Paso 1,500/2,000 | episodio=7 | epsilon=0.100 | loss=0.0026 | Q=0.246
Paso 1,750/2,000 | episodio=9 | epsilon=0.100 | loss=0.0049 | Q=0.373
Paso 2,000/2,000 | episodio=10 | epsilon=0.100 | loss=0.0013 | Q=0.336

EVALUACIÓN | paso=2,000 | promedio=80.00 | mediana=80.00 | máximo=80.00


SMOKE TEST DQN + PER FINALIZADO
Mejor promedio de evaluación: 80.00


,paso_global,loss,error_td_promedio,beta_per,peso_importancia_promedio,tamano_buffer
0,250,0.005354,0.083589,0.4744,0.453879,250
1,500,0.013343,0.214829,0.5500,0.373648,500
2,750,0.002121,0.115667,0.6244,0.180380,750
3,1000,0.003941,0.129842,0.7000,0.312108,1000
4,1250,0.010394,0.227189,0.7744,0.377781,1250
5,1500,0.002597,0.148887,0.8500,0.194636,1500
6,1750,0.004911,0.191007,0.9244,0.289631,1750
7,2000,0.001277,0.171572,1.0000,0.159696,2000


## Entrenamiento DQN + PER

In [6]:
# Liberar objetos del smoke test
objetos_temporales = [
    "resultado_smoke_per",
    "df_updates_smoke",
]

for nombre in objetos_temporales:
    globals().pop(nombre, None)

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

config_dqn_per = ConfigDQN(
    nombre_experimento="v5_dqn_per",
    total_pasos=1_000_000,

    gamma=0.99,
    learning_rate=1e-4,
    batch_size=32,
    frecuencia_entrenamiento=4,

    capacidad_buffer=20_000,
    inicio_entrenamiento=10_000,
    frecuencia_actualizacion_target=10_000,
    gradient_clip=10.0,

    epsilon_inicial=1.0,
    epsilon_final=0.1,
    pasos_decay_epsilon=250_000,

    # Configuración de PER
    usar_per=True,
    per_alpha=0.6,
    per_beta_inicial=0.4,
    per_beta_final=1.0,
    per_pasos_beta=500_000,
    per_epsilon=1e-6,

    frecuencia_evaluacion=50_000,
    episodios_evaluacion=5,
    seed_evaluacion=1_000,
    frecuencia_log=1_000,

    terminal_on_life_loss=True,
    clip_reward=True,
)

resultado_dqn_per = entrenar_dqn(
    config=config_dqn_per,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
)

print("\nENTRENAMIENTO DQN + PER FINALIZADO")
print(
    "Mejor promedio de evaluación: "
    f"{resultado_dqn_per['mejor_promedio_evaluacion']:.2f}"
)

Replay buffer: priorizado | alpha=0.6 | beta inicial=0.4
Paso 10,000/1,000,000 | episodio=49 | epsilon=0.964 | loss=0.0328 | Q=0.005
Paso 11,000/1,000,000 | episodio=52 | epsilon=0.960 | loss=0.0051 | Q=0.111
Paso 12,000/1,000,000 | episodio=58 | epsilon=0.957 | loss=0.0440 | Q=0.117
Paso 13,000/1,000,000 | episodio=63 | epsilon=0.953 | loss=0.0303 | Q=0.159
Paso 14,000/1,000,000 | episodio=68 | epsilon=0.950 | loss=0.0087 | Q=0.136
Paso 15,000/1,000,000 | episodio=72 | epsilon=0.946 | loss=0.0126 | Q=0.144
Paso 16,000/1,000,000 | episodio=78 | epsilon=0.942 | loss=0.0237 | Q=0.106
Paso 17,000/1,000,000 | episodio=83 | epsilon=0.939 | loss=0.0010 | Q=0.113
Paso 18,000/1,000,000 | episodio=88 | epsilon=0.935 | loss=0.0114 | Q=0.106
Paso 19,000/1,000,000 | episodio=93 | epsilon=0.932 | loss=0.0035 | Q=0.132
Paso 20,000/1,000,000 | episodio=99 | epsilon=0.928 | loss=0.0092 | Q=0.167
Paso 21,000/1,000,000 | episodio=102 | epsilon=0.924 | loss=0.0167 | Q=0.306
Paso 22,000/1,000,000 | episod

## Evaluaciones

In [7]:
ruta_evaluaciones_per = Path(
    "../logs/entrenamientos/v5_dqn_per/evaluaciones.csv"
)

df_evaluaciones_per = pd.read_csv(
    ruta_evaluaciones_per
)

df_evaluaciones_per = (
    df_evaluaciones_per
    .sort_values("paso_global")
    .reset_index(drop=True)
)

display(df_evaluaciones_per)

mejor_evaluacion_per = df_evaluaciones_per.loc[
    df_evaluaciones_per["promedio"].idxmax()
]

print("\nMEJOR EVALUACIÓN DE DQN + PER")
print(
    f"Paso: "
    f"{int(mejor_evaluacion_per['paso_global']):,}"
)
print(
    f"Promedio: "
    f"{mejor_evaluacion_per['promedio']:.2f}"
)
print(
    f"Mediana: "
    f"{mejor_evaluacion_per['mediana']:.2f}"
)
print(
    f"Desviación: "
    f"{mejor_evaluacion_per['desviacion']:.2f}"
)
print(
    f"Mínimo: "
    f"{mejor_evaluacion_per['minimo']:.2f}"
)
print(
    f"Máximo: "
    f"{mejor_evaluacion_per['maximo']:.2f}"
)

ruta_mejor_per = Path(
    "../models/v5_dqn_per/mejor_modelo.pt"
)

print(
    "\nCheckpoint del mejor modelo: "
    f"{'OK' if ruta_mejor_per.exists() else 'NO ENCONTRADO'}"
)

,paso_global,promedio,mediana,desviacion,minimo,maximo
0,50000,121.0,60.0,130.015384,40.0,380.0
1,100000,142.0,100.0,119.021007,55.0,375.0
2,150000,160.0,155.0,30.659419,120.0,215.0
3,200000,282.0,215.0,135.889661,130.0,480.0
4,250000,250.0,270.0,84.616783,140.0,370.0
5,300000,295.0,325.0,71.763500,210.0,395.0
6,350000,334.0,270.0,120.764233,240.0,570.0
7,400000,302.0,215.0,139.985714,155.0,525.0
8,450000,329.0,335.0,48.928519,245.0,395.0
9,500000,320.0,300.0,138.419652,120.0,550.0



MEJOR EVALUACIÓN DE DQN + PER
Paso: 700,000
Promedio: 344.00
Mediana: 405.00
Desviación: 138.69
Mínimo: 170.00
Máximo: 515.00

Checkpoint del mejor modelo: OK


In [9]:
import evaluation
importlib.reload(evaluation)

from evaluation import evaluar_modelo

In [10]:
N_EPISODIOS_EVALUACION = 30
SEMILLA_BASE_EVALUACION = 42

checkpoint_per = torch.load(
    ruta_mejor_per,
    map_location=device,
    weights_only=True,
)

mejor_dqn_per = DQN(
    n_acciones=6
).to(device)

mejor_dqn_per.load_state_dict(
    checkpoint_per["modelo_online_state_dict"]
)

mejor_dqn_per.eval()

print(
    f"Checkpoint cargado desde el paso: "
    f"{checkpoint_per['paso']:,}"
)

print(
    "Total de pasos registrado en la configuración: "
    f"{checkpoint_per['config']['total_pasos']:,}"
)

print(
    f"\nEvaluando DQN + PER durante "
    f"{N_EPISODIOS_EVALUACION} episodios...\n"
)

resultados_dqn_per, resumen_dqn_per = evaluar_modelo(
    modelo=mejor_dqn_per,
    config=config_dqn_per,
    device=device,
    n_episodios=N_EPISODIOS_EVALUACION,
    seed_base=SEMILLA_BASE_EVALUACION,
)

df_dqn_per = pd.DataFrame(
    resultados_dqn_per
)

df_dqn_per["agente"] = "DQN + PER"

df_dqn_per = df_dqn_per[
    [
        "agente",
        "episodio",
        "seed",
        "recompensa_total",
        "pasos",
        "terminated",
        "truncated",
    ]
]

print("RESUMEN DE 30 EPISODIOS")

for metrica, valor in resumen_dqn_per.items():
    print(f"{metrica}: {valor:.2f}")

display(df_dqn_per.head(10))

Checkpoint cargado desde el paso: 700,000
Total de pasos registrado en la configuración: 1,000,000

Evaluando DQN + PER durante 30 episodios...

RESUMEN DE 30 EPISODIOS
promedio: 282.00
mediana: 282.50
desviacion: 92.95
minimo: 155.00
maximo: 540.00


,agente,episodio,seed,recompensa_total,pasos,terminated,truncated
0,DQN + PER,0,42,295.0,711,True,False
1,DQN + PER,1,43,280.0,509,True,False
2,DQN + PER,2,44,315.0,618,True,False
3,DQN + PER,3,45,240.0,555,True,False
4,DQN + PER,4,46,245.0,534,True,False
5,DQN + PER,5,47,230.0,528,True,False
6,DQN + PER,6,48,165.0,367,True,False
7,DQN + PER,7,49,400.0,569,True,False
8,DQN + PER,8,50,250.0,522,True,False
9,DQN + PER,9,51,325.0,740,True,False


## Guardar y Comparar

In [11]:
ruta_resultados_per = Path(
    "../logs/evaluacion_dqn_per.csv"
)

df_dqn_per.to_csv(
    ruta_resultados_per,
    index=False,
)

df_dqn_vanilla = pd.read_csv(
    "../logs/evaluacion_dqn_vanilla.csv"
)

df_double_dqn = pd.read_csv(
    "../logs/evaluacion_double_dqn.csv"
)

df_dueling_dqn = pd.read_csv(
    "../logs/evaluacion_dueling_double_dqn.csv"
)

df_modelos_per = pd.concat(
    [
        df_dqn_vanilla,
        df_double_dqn,
        df_dueling_dqn,
        df_dqn_per,
    ],
    ignore_index=True,
)

resumen_modelos_per = (
    df_modelos_per
    .groupby("agente")
    .agg(
        episodios=("recompensa_total", "count"),
        promedio=("recompensa_total", "mean"),
        mediana=("recompensa_total", "median"),
        desviacion=("recompensa_total", "std"),
        minimo=("recompensa_total", "min"),
        maximo=("recompensa_total", "max"),
        pasos_promedio=("pasos", "mean"),
    )
    .round(2)
    .reset_index()
)

display(resumen_modelos_per)

comparacion_per = (
    pd.concat(
        [
            df_dqn_vanilla,
            df_dqn_per,
        ],
        ignore_index=True,
    )
    .pivot(
        index="seed",
        columns="agente",
        values="recompensa_total",
    )
    .dropna()
)

diferencia = (
    comparacion_per["DQN + PER"]
    - comparacion_per["DQN vanilla"]
)

print(
    "Victorias DQN vanilla:",
    (diferencia < 0).sum(),
)

print(
    "Victorias DQN + PER:",
    (diferencia > 0).sum(),
)

print(
    "Empates:",
    (diferencia == 0).sum(),
)

print(
    f"\nResultados guardados en: "
    f"{ruta_resultados_per}"
)

,agente,episodios,promedio,mediana,desviacion,minimo,maximo,pasos_promedio
0,DQN + PER,30,282.00,282.5,94.54,155.0,540.0,569.20
1,DQN vanilla,30,406.67,377.5,139.11,215.0,670.0,705.80
2,Double DQN,30,283.33,262.5,154.85,105.0,925.0,614.73
3,Dueling Double DQN,30,263.33,215.0,112.84,155.0,580.0,597.50


Victorias DQN vanilla: 22
Victorias DQN + PER: 8
Empates: 0

Resultados guardados en: ../logs/evaluacion_dqn_per.csv


## Prueba acumulador n-step

In [12]:
importlib.reload(replay_buffer)

from replay_buffer import NStepAccumulator

acumulador_prueba = NStepAccumulator(
    n_step=3,
    gamma=0.9,
)

transiciones_generadas = []

for paso in range(4):
    observacion = np.array(
        [paso],
        dtype=np.uint8,
    )

    siguiente_observacion = np.array(
        [paso + 1],
        dtype=np.uint8,
    )

    finalizado = paso == 3

    nuevas_transiciones = acumulador_prueba.agregar(
        observacion=observacion,
        accion=paso,
        recompensa=1.0,
        siguiente_observacion=siguiente_observacion,
        finalizado=finalizado,
    )

    transiciones_generadas.extend(
        nuevas_transiciones
    )

print(
    f"Transiciones generadas: "
    f"{len(transiciones_generadas)}"
)

for indice, transicion in enumerate(
    transiciones_generadas
):
    (
        observacion,
        accion,
        recompensa,
        siguiente_observacion,
        finalizado,
    ) = transicion

    print(
        f"{indice}: "
        f"obs={observacion.item()} | "
        f"acción={accion} | "
        f"recompensa={recompensa:.3f} | "
        f"siguiente={siguiente_observacion.item()} | "
        f"finalizado={finalizado}"
    )

recompensas_obtenidas = np.array(
    [
        transicion[2]
        for transicion in transiciones_generadas
    ]
)

recompensas_esperadas = np.array(
    [
        1 + 0.9 + 0.9**2,
        1 + 0.9 + 0.9**2,
        1 + 0.9,
        1.0,
    ]
)

assert len(transiciones_generadas) == 4

assert np.allclose(
    recompensas_obtenidas,
    recompensas_esperadas,
)

assert len(acumulador_prueba) == 0

print(
    "\nEl acumulador n-step funciona correctamente."
)

Transiciones generadas: 4
0: obs=0 | acción=0 | recompensa=2.710 | siguiente=3 | finalizado=False
1: obs=1 | acción=1 | recompensa=2.710 | siguiente=4 | finalizado=True
2: obs=2 | acción=2 | recompensa=1.900 | siguiente=4 | finalizado=True
3: obs=3 | acción=3 | recompensa=1.000 | siguiente=4 | finalizado=True

El acumulador n-step funciona correctamente.


In [13]:
import gc
import importlib
import torch

import replay_buffer
import train
import models

importlib.reload(replay_buffer)
importlib.reload(train)
importlib.reload(models)

from models import DQN
from train import ConfigDQN, entrenar_dqn

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

config_smoke_n_step = ConfigDQN(
    nombre_experimento="smoke_test_dqn_per_3step",
    total_pasos=2_000,

    gamma=0.99,
    n_step=3,

    usar_per=True,
    per_alpha=0.6,
    per_beta_inicial=0.4,
    per_beta_final=1.0,
    per_pasos_beta=2_000,
    per_epsilon=1e-6,

    capacidad_buffer=2_000,
    inicio_entrenamiento=200,
    batch_size=32,
    frecuencia_entrenamiento=4,
    frecuencia_actualizacion_target=500,

    epsilon_inicial=1.0,
    epsilon_final=0.1,
    pasos_decay_epsilon=1_000,

    frecuencia_log=250,
    frecuencia_evaluacion=2_000,
    episodios_evaluacion=1,
)

resultado_smoke_n_step = entrenar_dqn(
    config=config_smoke_n_step,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
)

print("\nSMOKE TEST PER + 3-STEP FINALIZADO")
print(
    "Mejor promedio de evaluación: "
    f"{resultado_smoke_n_step['mejor_promedio_evaluacion']:.2f}"
)

Replay buffer: priorizado | alpha=0.6 | beta inicial=0.4
Retorno utilizado: 3-step
Paso 250/2,000 | episodio=1 | epsilon=0.775 | loss=0.0093 | Q=0.062
Paso 500/2,000 | episodio=1 | epsilon=0.550 | loss=0.0222 | Q=0.154
Paso 750/2,000 | episodio=2 | epsilon=0.325 | loss=0.0065 | Q=0.288
Paso 1,000/2,000 | episodio=3 | epsilon=0.100 | loss=0.0168 | Q=0.367
Paso 1,250/2,000 | episodio=4 | epsilon=0.100 | loss=0.0094 | Q=0.480
Paso 1,500/2,000 | episodio=6 | epsilon=0.100 | loss=0.0045 | Q=0.507
Paso 1,750/2,000 | episodio=7 | epsilon=0.100 | loss=0.0057 | Q=0.572
Paso 2,000/2,000 | episodio=8 | epsilon=0.100 | loss=0.0049 | Q=0.557

EVALUACIÓN | paso=2,000 | promedio=75.00 | mediana=75.00 | máximo=75.00


SMOKE TEST PER + 3-STEP FINALIZADO
Mejor promedio de evaluación: 75.00


## Entrenamiento de DQN + PER + 3-step

In [14]:
globals().pop(
    "resultado_smoke_n_step",
    None,
)

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

config_dqn_per_3step = ConfigDQN(
    nombre_experimento="v6_dqn_per_3step",
    total_pasos=1_000_000,

    gamma=0.99,
    n_step=3,
    learning_rate=1e-4,
    batch_size=32,
    frecuencia_entrenamiento=4,

    capacidad_buffer=20_000,
    inicio_entrenamiento=10_000,
    frecuencia_actualizacion_target=10_000,
    gradient_clip=10.0,

    epsilon_inicial=1.0,
    epsilon_final=0.1,
    pasos_decay_epsilon=250_000,

    usar_per=True,
    per_alpha=0.6,
    per_beta_inicial=0.4,
    per_beta_final=1.0,
    per_pasos_beta=500_000,
    per_epsilon=1e-6,

    frecuencia_evaluacion=50_000,
    episodios_evaluacion=5,
    seed_evaluacion=1_000,
    frecuencia_log=1_000,

    terminal_on_life_loss=True,
    clip_reward=True,
)

resultado_dqn_per_3step = entrenar_dqn(
    config=config_dqn_per_3step,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
)

print(
    "\nENTRENAMIENTO DQN + PER + 3-STEP FINALIZADO"
)

print(
    "Mejor promedio de evaluación: "
    f"{resultado_dqn_per_3step['mejor_promedio_evaluacion']:.2f}"
)

Replay buffer: priorizado | alpha=0.6 | beta inicial=0.4
Retorno utilizado: 3-step
Paso 10,000/1,000,000 | episodio=49 | epsilon=0.964 | loss=0.0550 | Q=0.006
Paso 11,000/1,000,000 | episodio=53 | epsilon=0.960 | loss=0.0135 | Q=0.106
Paso 12,000/1,000,000 | episodio=59 | epsilon=0.957 | loss=0.0399 | Q=0.167
Paso 13,000/1,000,000 | episodio=66 | epsilon=0.953 | loss=0.0353 | Q=0.174
Paso 14,000/1,000,000 | episodio=71 | epsilon=0.950 | loss=0.0331 | Q=0.178
Paso 15,000/1,000,000 | episodio=77 | epsilon=0.946 | loss=0.0496 | Q=0.188
Paso 16,000/1,000,000 | episodio=82 | epsilon=0.942 | loss=0.0328 | Q=0.127
Paso 17,000/1,000,000 | episodio=86 | epsilon=0.939 | loss=0.0157 | Q=0.164
Paso 18,000/1,000,000 | episodio=92 | epsilon=0.935 | loss=0.0289 | Q=0.230
Paso 19,000/1,000,000 | episodio=98 | epsilon=0.932 | loss=0.0225 | Q=0.171
Paso 20,000/1,000,000 | episodio=103 | epsilon=0.928 | loss=0.0220 | Q=0.208
Paso 21,000/1,000,000 | episodio=108 | epsilon=0.924 | loss=0.0228 | Q=0.422
Pas

In [15]:
from pathlib import Path
import pandas as pd

ruta_evaluaciones_3step = Path(
    "../logs/entrenamientos/"
    "v6_dqn_per_3step/evaluaciones.csv"
)

df_evaluaciones_3step = pd.read_csv(
    ruta_evaluaciones_3step
)

df_evaluaciones_3step = (
    df_evaluaciones_3step
    .sort_values("paso_global")
    .reset_index(drop=True)
)

display(df_evaluaciones_3step)

mejor_evaluacion_3step = (
    df_evaluaciones_3step.loc[
        df_evaluaciones_3step["promedio"].idxmax()
    ]
)

print("\nMEJOR EVALUACIÓN DE DQN + PER + 3-STEP")
print(
    f"Paso: "
    f"{int(mejor_evaluacion_3step['paso_global']):,}"
)
print(
    f"Promedio: "
    f"{mejor_evaluacion_3step['promedio']:.2f}"
)
print(
    f"Mediana: "
    f"{mejor_evaluacion_3step['mediana']:.2f}"
)
print(
    f"Desviación: "
    f"{mejor_evaluacion_3step['desviacion']:.2f}"
)
print(
    f"Mínimo: "
    f"{mejor_evaluacion_3step['minimo']:.2f}"
)
print(
    f"Máximo: "
    f"{mejor_evaluacion_3step['maximo']:.2f}"
)

ruta_mejor_3step = Path(
    "../models/v6_dqn_per_3step/mejor_modelo.pt"
)

print(
    "\nCheckpoint del mejor modelo: "
    f"{'OK' if ruta_mejor_3step.exists() else 'NO ENCONTRADO'}"
)

,paso_global,promedio,mediana,desviacion,minimo,maximo
0,50000,224.0,215.0,19.849433,200.0,250.0
1,100000,138.0,135.0,41.904654,70.0,195.0
2,150000,248.0,270.0,38.807216,180.0,285.0
3,200000,314.0,250.0,165.571737,190.0,640.0
4,250000,333.0,300.0,115.697882,195.0,545.0
5,300000,404.0,285.0,206.746221,225.0,770.0
6,350000,254.0,265.0,51.029403,170.0,325.0
7,400000,431.0,330.0,177.380946,320.0,780.0
8,450000,412.0,435.0,70.327804,320.0,500.0
9,500000,347.0,280.0,118.473626,235.0,510.0



MEJOR EVALUACIÓN DE DQN + PER + 3-STEP
Paso: 700,000
Promedio: 475.00
Mediana: 510.00
Desviación: 123.94
Mínimo: 280.00
Máximo: 610.00

Checkpoint del mejor modelo: OK
